***

* [总目录](../0_Introduction/0_introduction.ipynb)
* [术语表](../0_Introduction/1_glossary.ipynb)
* [7. 观测系统](7_0_introduction.ipynb)
    * 上一节： [7.2 射电干涉测量方程](7_2_rime.ipynb)
    * 下一节： [7.4 数字相关器](7_4_digital.ipynb)

***

导入标准模块:

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

plt.rcParams['figure.max_open_warning'] = 0

NOTEBOOK_DIR = Path('7_Observing_Systems') if Path('7_Observing_Systems').exists() else Path('.')
NOTEBOOK_DIR = NOTEBOOK_DIR.resolve()
STYLE_PATH = NOTEBOOK_DIR.parent / 'style' / 'course.css'
TOGGLE_PATH = NOTEBOOK_DIR.parent / 'style' / 'code_toggle.html'

try:
    from IPython.display import HTML
except ImportError:
    def HTML(*args, **kwargs):
        return None

if STYLE_PATH.exists():
    try:
        HTML(STYLE_PATH.read_text(encoding='utf-8'))
    except Exception:
        pass

本节以下示例全部使用 notebook 内生成的频谱、噪声链路和混频信号，因此不依赖旧版外部图片或 `scipy`。重点将围绕一条完整的模拟前端逻辑展开：源频谱如何被 `G-Jones` 与 `B-Jones` 改写，系统温度如何决定灵敏度，为什么 LNA 必须尽早出现，以及滤波和外差混频如何把射频信号变成适合 ADC 的中频信号。

In [ ]:
if TOGGLE_PATH.exists():
    try:
        HTML(TOGGLE_PATH.read_text(encoding='utf-8'))
    except Exception:
        pass

***

## 7.3 模拟电子学（G-Jones 与 B-Jones）

从 RIME 的角度看，模拟电子链路最常体现在方向无关 Jones 项里。很多教材会把整个模拟前端粗略记成一个总增益矩阵，但在工程上，把它再拆成“主要随时间变化的增益项”与“主要随频率变化的带通项”往往更有用：

$$
\mathbf{J}_{\mathrm{analog}}(t,\nu) \approx \mathbf{G}(t)\,\mathbf{B}(\nu).
$$

这里的 $\mathbf{G}$-Jones 主要对应慢时间尺度上的增益漂移，例如温度变化、电子学稳定性和本振漂移；而 $\mathbf{B}$-Jones 主要对应频率响应，也就是带通、滤波器边缘、波导/电缆起伏以及模拟器件的频率依赖增益。

本节并不试图把所有模拟器件都讲成一本电子学教材，而是聚焦于几个对射电干涉数据处理最关键的结论：

1. 模拟链路首先决定了我们能否把极弱天空信号抬升到可数字化的尺度；
2. 系统温度决定了灵敏度基线，后续任何校准都无法“凭空造出”被噪声淹没的信息；
3. 前端级联次序极其重要，尤其是低噪声放大器（LNA）的位置；
4. 模拟滤波和外差混频共同决定了进入 ADC 之前的有效频带和中频布局。

In [ ]:
def source_spectrum(freqs_hz, I0=1.0, nu0_hz=1.4e9, alpha=-0.7):
    return I0 * (freqs_hz / nu0_hz) ** alpha



def smooth_bandpass(freqs_hz, center_hz, width_hz, ripple=0.03, slope=0.10,
                    notch_hz=None, notch_depth=0.0, notch_width_hz=1.0):
    x = (freqs_hz - center_hz) / width_hz
    envelope = np.exp(-0.5 * x ** 8)
    tilt = 1.0 + slope * x
    ripple_term = 1.0 + ripple * np.cos(8.0 * np.pi * x)
    bandpass = np.clip(envelope * tilt * ripple_term, 0.0, None)
    if notch_hz is not None and notch_depth > 0.0:
        notch = 1.0 - notch_depth * np.exp(-0.5 * ((freqs_hz - notch_hz) / notch_width_hz) ** 2)
        bandpass *= np.clip(notch, 0.0, None)
    return bandpass



def gain_drift(times_s, amp=0.04, period_s=1200.0, phase=0.0):
    return 1.0 + amp * np.sin(2.0 * np.pi * times_s / period_s + phase)



def radiometer_sigma(Tsys_K, bandwidth_Hz, integ_s):
    return Tsys_K / np.sqrt(bandwidth_Hz * integ_s)



def sefd_jy(Tsys_K, aeff_m2, efficiency=1.0):
    k_B = 1.380649e-23
    return 2.0 * k_B * Tsys_K / (efficiency * aeff_m2) / 1e-26



def db_power(x):
    x = np.asarray(x, dtype=float)
    return 10.0 * np.log10(np.clip(x, 1e-12, None))



def power_gain_from_db(gain_db):
    return 10.0 ** (gain_db / 10.0)



def cascaded_receiver_temperature(stages):
    gain_product = 1.0
    total_temp = 0.0
    cumulative_gain = []
    cumulative_temp = []
    for stage in stages:
        total_temp += stage['T_K'] / gain_product
        gain_product *= stage['gain_power']
        cumulative_gain.append(gain_product)
        cumulative_temp.append(total_temp)
    return total_temp, np.array(cumulative_gain), np.array(cumulative_temp)



def fft_spectrum(signal, dt_s):
    spec = np.fft.rfft(signal)
    freq = np.fft.rfftfreq(signal.size, d=dt_s)
    return freq, np.abs(spec)



def lowpass_in_frequency_domain(signal, dt_s, cutoff_hz):
    spec = np.fft.rfft(signal)
    freq = np.fft.rfftfreq(signal.size, d=dt_s)
    spec[freq > cutoff_hz] = 0.0
    return np.fft.irfft(spec, n=signal.size)

### 7.3.1 理想源、G-Jones 与 B-Jones

先从一个理想化的窄带连续谱源出发。设其 Stokes $I$ 频谱满足幂律

$$
I(\nu) = I_0\left(\frac{\nu}{\nu_0}\right)^{\alpha},
$$

且在短时间内本征流量不变。若接收机没有任何频率响应和时间漂移，那么动态频谱只会忠实反映这个源本身。现实中，模拟链路通常会同时引入：

* 随时间缓慢变化的增益漂移 $G(t)$；
* 随频率变化的带通与陷波响应 $B(\nu)$；
* 两个正交极化通道之间略有差异的幅度标定。

下面用一个简化的双极化例子直观看看这种分解。

In [ ]:
nChannels = 256
nTimes = 240
freqs = np.linspace(0.9e9, 1.7e9, nChannels)
times = np.linspace(0.0, 1800.0, nTimes)

idealSpectrum = source_spectrum(freqs, I0=1.2, nu0_hz=1.4e9, alpha=-0.65)
Bx = smooth_bandpass(freqs, center_hz=1.30e9, width_hz=3.6e8, ripple=0.025, slope=0.08,
                     notch_hz=1.52e9, notch_depth=0.85, notch_width_hz=1.2e7)
By = smooth_bandpass(freqs, center_hz=1.31e9, width_hz=3.4e8, ripple=0.03, slope=-0.05,
                     notch_hz=1.52e9, notch_depth=0.70, notch_width_hz=1.4e7)
Gx = gain_drift(times, amp=0.035, period_s=1500.0, phase=0.2)
Gy = gain_drift(times, amp=0.025, period_s=1100.0, phase=1.0)

observedXX = np.outer(Gx, idealSpectrum * Bx)
observedYY = np.outer(Gy, idealSpectrum * By)

print(f'XX 动态频谱最小/最大值                 = {observedXX.min():.3f} / {observedXX.max():.3f}')
print(f'YY 动态频谱最小/最大值                 = {observedYY.min():.3f} / {observedYY.max():.3f}')
print(f'中频段 XX/YY 响应比的中位数            = {np.median(observedXX[:, 120] / observedYY[:, 120]):.3f}')

fig, axes = plt.subplots(2, 3, figsize=(15, 9))

axes[0, 0].plot(freqs / 1e9, idealSpectrum, linewidth=2)
axes[0, 0].set_title('Ideal source spectrum')
axes[0, 0].set_xlabel('Frequency [GHz]')
axes[0, 0].set_ylabel('Flux [arb.]')
axes[0, 0].grid(alpha=0.25)

axes[0, 1].plot(freqs / 1e9, Bx, linewidth=2, label='Bx')
axes[0, 1].plot(freqs / 1e9, By, linewidth=2, label='By')
axes[0, 1].set_title('Bandpass terms B(nu)')
axes[0, 1].set_xlabel('Frequency [GHz]')
axes[0, 1].set_ylabel('Gain')
axes[0, 1].grid(alpha=0.25)
axes[0, 1].legend()

axes[0, 2].plot(times / 60.0, Gx, linewidth=2, label='Gx')
axes[0, 2].plot(times / 60.0, Gy, linewidth=2, label='Gy')
axes[0, 2].set_title('Time-dependent gain drift G(t)')
axes[0, 2].set_xlabel('Time [min]')
axes[0, 2].set_ylabel('Gain')
axes[0, 2].grid(alpha=0.25)
axes[0, 2].legend()

im0 = axes[1, 0].imshow(observedXX, aspect='auto', origin='lower',
                        extent=[freqs[0] / 1e9, freqs[-1] / 1e9, times[0] / 60.0, times[-1] / 60.0])
axes[1, 0].set_title('Observed XX dynamic spectrum')
axes[1, 0].set_xlabel('Frequency [GHz]')
axes[1, 0].set_ylabel('Time [min]')
fig.colorbar(im0, ax=axes[1, 0], shrink=0.85)

im1 = axes[1, 1].imshow(observedYY, aspect='auto', origin='lower',
                        extent=[freqs[0] / 1e9, freqs[-1] / 1e9, times[0] / 60.0, times[-1] / 60.0])
axes[1, 1].set_title('Observed YY dynamic spectrum')
axes[1, 1].set_xlabel('Frequency [GHz]')
axes[1, 1].set_ylabel('Time [min]')
fig.colorbar(im1, ax=axes[1, 1], shrink=0.85)

axes[1, 2].plot(freqs / 1e9, observedXX[nTimes // 2], linewidth=2, label='XX at mid-time')
axes[1, 2].plot(freqs / 1e9, observedYY[nTimes // 2], linewidth=2, label='YY at mid-time')
axes[1, 2].set_title('One time slice after G and B')
axes[1, 2].set_xlabel('Frequency [GHz]')
axes[1, 2].set_ylabel('Observed flux [arb.]')
axes[1, 2].grid(alpha=0.25)
axes[1, 2].legend(fontsize=9)

plt.tight_layout()

这里最重要的工程结论是：即使天空源本身在时间上几乎稳定，模拟链路也会让我们看到带有时间漂移和频率起伏的动态频谱。对校准而言，这意味着至少要区分两类问题：

* 是天空真的在变，还是 $G(t)$ 在漂移；
* 是源的真实谱结构，还是 $B(\nu)$ 的带通形状在起作用。

也正因如此，很多数据处理流程会把时间依赖增益和频率依赖带通分开求解。这个分解不是纯粹的记号偏好，而是由模拟链路的物理结构直接推动出来的。

### 7.3.2 系统噪声、辐射计方程与 SEFD

无论模拟链路设计得多好，系统总会引入噪声。把各种噪声源合并后，我们通常用系统温度表示整体噪声预算：

$$
T_{\mathrm{sys}} = T_{\mathrm{sky}} + T_{\mathrm{atm}} + T_{\mathrm{spill}} + T_{\mathrm{rx}} + \cdots
$$

在理想辐射计近似下，热噪声的不确定度满足

$$
\sigma_T = \frac{T_{\mathrm{sys}}}{\sqrt{\Delta\nu\,\tau}},
$$

其中 $\Delta\nu$ 是有效带宽，$\tau$ 是积分时间。这个公式浓缩了射电观测里最基本的一条灵敏度规律：增加带宽和积分时间只能按平方根改善噪声，而降低系统温度则是线性获益。

另一种常见的灵敏度表征是系统等效通量密度（SEFD）：

$$
\mathrm{SEFD} = \frac{2k_B T_{\mathrm{sys}}}{\eta A_{\mathrm{eff}}}.
$$

它把噪声温度换算成等效的天空通量密度，是把仪器噪声与天体信号直接放在同一单位下比较的自然方式。

In [ ]:
Tsys = 55.0
integrationTimes = np.logspace(0.0, 4.0, 200)  # 1 s to 10^4 s
bandwidths = [1e6, 10e6, 100e6]

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for bandwidth in bandwidths:
    axes[0].loglog(integrationTimes, radiometer_sigma(Tsys, bandwidth, integrationTimes), linewidth=2,
                   label=f'{bandwidth / 1e6:.0f} MHz')
axes[0].set_title('Radiometer equation')
axes[0].set_xlabel('Integration time [s]')
axes[0].set_ylabel(r'$\sigma_T$ [K]')
axes[0].grid(alpha=0.25, which='both')
axes[0].legend()

noiseTerms = {
    'Sky': 11.0,
    'Atmosphere': 4.5,
    'Spillover': 7.0,
    'Receiver': 25.0,
    'Other': 7.5,
}
axes[1].bar(noiseTerms.keys(), noiseTerms.values(), color=['#4c72b0', '#55a868', '#c44e52', '#8172b2', '#ccb974'])
axes[1].set_title('Example system-temperature budget')
axes[1].set_ylabel('Noise temperature [K]')
axes[1].grid(axis='y', alpha=0.25)

plt.tight_layout()

print('辐射计方程示例：')
for bandwidth in bandwidths:
    sigma_10s = radiometer_sigma(Tsys, bandwidth, 10.0)
    print(f'  bandwidth = {bandwidth / 1e6:6.0f} MHz -> sigma_T(10 s) = {sigma_10s:.5f} K')

exampleSEFD = sefd_jy(Tsys_K=55.0, aeff_m2=140.0, efficiency=0.72)
print()
print(f'当 Tsys = 55 K, eta*Aeff = 0.72 x 140 m^2 时，SEFD ≈ {exampleSEFD:.1f} Jy')

从这个例子可以直接看到两件非常实用的事：

* 把带宽从 1 MHz 提到 100 MHz，会让热噪声下降一个数量级，但前提是整个频带都真正可用；
* 如果系统温度预算中接收机噪声占了很大比例，那么继续优化模拟前端往往比单纯延长观测时间更有效。

这也是为什么第 7 章要专门花精力讲模拟链路：它不仅决定“信号能不能被送进数字域”，还直接决定“送进去的信号有多脏”。

### 7.3.3 模拟前端级联与 LNA 为什么必须靠前

模拟前端不是单个器件，而是一串级联的增益和噪声源。对一个线性级联系统，后级噪声会被前级增益压低；反过来说，如果在高增益 LNA 之前先放入有损耗的无源部件，那么它们引入的噪声会被完整地暴露给后端，造成非常明显的灵敏度损失。

这就是经典 Friis 噪声级联公式的物理意义。若各级附加噪声温度为 $T_i$、前面各级功率增益为 $G_i$，则总接收机噪声温度近似为

$$
T_{\mathrm{rx}} = T_1 + \frac{T_2}{G_1} + \frac{T_3}{G_1G_2} + \cdots.
$$

下面直接比较两条前端方案：一条让高增益 LNA 尽早出现；另一条则在 LNA 之前先经过额外的温暖损耗。

In [ ]:
stagesLnaFirst = [
    {'name': 'Feed + waveguide loss', 'gain_db': -0.7, 'T_K': 22.0},
    {'name': 'Cryogenic LNA', 'gain_db': 35.0, 'T_K': 8.0},
    {'name': 'Cable + filter', 'gain_db': -2.0, 'T_K': 35.0},
    {'name': 'IF amplifier', 'gain_db': 20.0, 'T_K': 120.0},
]
stagesLossFirst = [
    {'name': 'Warm cable', 'gain_db': -2.0, 'T_K': 35.0},
    {'name': 'Cryogenic LNA', 'gain_db': 35.0, 'T_K': 8.0},
    {'name': 'IF amplifier', 'gain_db': 20.0, 'T_K': 120.0},
]

for stages in [stagesLnaFirst, stagesLossFirst]:
    for stage in stages:
        stage['gain_power'] = power_gain_from_db(stage['gain_db'])

rxLnaFirst, gainLnaFirst, tempLnaFirst = cascaded_receiver_temperature(stagesLnaFirst)
rxLossFirst, gainLossFirst, tempLossFirst = cascaded_receiver_temperature(stagesLossFirst)

print('级联接收机噪声温度：')
print(f'  LNA 尽早出现      -> T_rx = {rxLnaFirst:.2f} K, total gain = {db_power(gainLnaFirst[-1]):.1f} dB')
print(f'  先经过额外损耗     -> T_rx = {rxLossFirst:.2f} K, total gain = {db_power(gainLossFirst[-1]):.1f} dB')
print(f'  二者差值           -> ΔT_rx = {rxLossFirst - rxLnaFirst:.2f} K')

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

axes[0].step(np.arange(1, len(tempLnaFirst) + 1), tempLnaFirst, where='mid', linewidth=2.5, label='LNA first')
axes[0].step(np.arange(1, len(tempLossFirst) + 1), tempLossFirst, where='mid', linewidth=2.5, label='Loss first')
axes[0].set_title('Cumulative receiver temperature')
axes[0].set_xlabel('Stage index')
axes[0].set_ylabel(r'Equivalent $T_{rx}$ [K]')
axes[0].grid(alpha=0.25)
axes[0].legend()

stageLabels = ['LNA first', 'Loss first']
axes[1].bar(stageLabels, [rxLnaFirst, rxLossFirst], color=['#4c72b0', '#c44e52'])
axes[1].set_title('Front-end ordering matters')
axes[1].set_ylabel(r'Equivalent $T_{rx}$ [K]')
axes[1].grid(axis='y', alpha=0.25)

plt.tight_layout()

这个例子正好说明了为什么射电接收机设计里总强调“把低噪声、高增益放大器尽量靠前放”。一旦 LNA 先把天空信号抬起来，后面的滤波器、电缆和 IF 放大器即便不够理想，它们对总噪声预算的伤害也会被前级增益显著压低；反过来，如果在 LNA 前面先损失掉信号，那么后面再怎么放大，也只是把已经恶化的信噪比一起放大而已。

### 7.3.4 滤波器、带通与分贝表示

模拟前端的另一个核心任务，是把我们关心的频段挑出来，并尽量压制不想要的频段。工程上常常同时会遇到：

* 带通滤波器定义总体可用频带；
* 陷波器压制已知强干扰频段；
* 实际带通边缘并不陡峭，通带内还会有起伏和斜率。

这些频率响应常常会用分贝（dB）表示，因为它更方便表达大动态范围增益与衰减。对功率比而言，

$$
G_{\mathrm{dB}} = 10\log_{10} G_{\mathrm{power}}.
$$

下面的例子展示一个带有通带起伏和深陷波的模拟带通响应，以及它如何改写理想源谱。

In [ ]:
bandpassResponse = smooth_bandpass(
    freqs,
    center_hz=1.30e9,
    width_hz=3.5e8,
    ripple=0.03,
    slope=0.10,
    notch_hz=1.52e9,
    notch_depth=0.92,
    notch_width_hz=1.0e7,
)
filteredSpectrum = idealSpectrum * bandpassResponse

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].plot(freqs / 1e9, bandpassResponse, linewidth=2)
axes[0].set_title('Bandpass response (linear scale)')
axes[0].set_xlabel('Frequency [GHz]')
axes[0].set_ylabel('Gain')
axes[0].grid(alpha=0.25)

axes[1].plot(freqs / 1e9, db_power(bandpassResponse), linewidth=2, label='Bandpass in dB')
axes[1].plot(freqs / 1e9, db_power(filteredSpectrum / np.max(filteredSpectrum)), linewidth=2, label='Filtered source spectrum')
axes[1].set_title('Bandpass and filtered spectrum in dB')
axes[1].set_xlabel('Frequency [GHz]')
axes[1].set_ylabel('Relative power [dB]')
axes[1].grid(alpha=0.25)
axes[1].legend(fontsize=9)

plt.tight_layout()

print(f'1.52 GHz 陷波最深处的相对衰减约为 {db_power(np.min(bandpassResponse)):.1f} dB')
print(f'通带峰-谷起伏约为 {(db_power(np.max(bandpassResponse)) - db_power(np.median(bandpassResponse[(freqs>1.1e9)&(freqs<1.45e9)]))):.2f} dB')

从数据处理角度看，这类带通起伏正是后续带通校准要拟合和去除的对象。它们看似只是“光滑频谱上的小结构”，但如果不加校正，就会直接污染谱线观测、宽带连续谱拟合以及任何依赖频率平滑性的科学分析。

### 7.3.5 外差混频：把射频移到更易处理的中频

很多射电接收机不会直接在原始射频（RF）上做后续处理，而会先把信号与本振（LO）相乘，下变到中频（IF）。如果输入中包含频率 $\nu_{\mathrm{RF}}$，本振频率为 $\nu_{\mathrm{LO}}$，则混频后会同时出现差频和和频：

$$
\cos(2\pi \nu_{\mathrm{RF}} t)\cos(2\pi \nu_{\mathrm{LO}} t)
= \frac{1}{2}\cos(2\pi(\nu_{\mathrm{RF}}-\nu_{\mathrm{LO}})t)
+ \frac{1}{2}\cos(2\pi(\nu_{\mathrm{RF}}+\nu_{\mathrm{LO}})t).
$$

通过后续低通或带通滤波，只保留需要的差频部分，就能把高频 RF 变成更低、更容易采样和处理的 IF。下面用一组玩具频率直接演示这个过程。

In [ ]:
sampleRate = 400e6
sampleStep = 1.0 / sampleRate
nSamples = 8192
timeAxis = np.arange(nSamples) * sampleStep

rfSignal = np.cos(2.0 * np.pi * 92e6 * timeAxis) + 0.6 * np.cos(2.0 * np.pi * 118e6 * timeAxis)
loSignal = np.cos(2.0 * np.pi * 100e6 * timeAxis)
mixedSignal = rfSignal * loSignal
ifSignal = lowpass_in_frequency_domain(mixedSignal, sampleStep, cutoff_hz=30e6)

freqRF, ampRF = fft_spectrum(rfSignal, sampleStep)
freqMixed, ampMixed = fft_spectrum(mixedSignal, sampleStep)
freqIF, ampIF = fft_spectrum(ifSignal, sampleStep)

fig, axes = plt.subplots(2, 2, figsize=(14, 8))
axes[0, 0].plot(timeAxis[:800] * 1e6, rfSignal[:800], linewidth=1.4)
axes[0, 0].set_title('RF waveform (time domain)')
axes[0, 0].set_xlabel('Time [microsecond]')
axes[0, 0].set_ylabel('Amplitude')
axes[0, 0].grid(alpha=0.25)

axes[0, 1].plot(freqRF / 1e6, ampRF, linewidth=1.5)
axes[0, 1].set_xlim(0, 160)
axes[0, 1].set_title('RF spectrum')
axes[0, 1].set_xlabel('Frequency [MHz]')
axes[0, 1].set_ylabel('Amplitude')
axes[0, 1].grid(alpha=0.25)

axes[1, 0].plot(freqMixed / 1e6, ampMixed, linewidth=1.5, label='After mixing')
axes[1, 0].plot(freqIF / 1e6, ampIF, linewidth=1.8, label='After low-pass', alpha=0.85)
axes[1, 0].set_xlim(0, 220)
axes[1, 0].set_title('Mixed spectrum and selected IF')
axes[1, 0].set_xlabel('Frequency [MHz]')
axes[1, 0].set_ylabel('Amplitude')
axes[1, 0].grid(alpha=0.25)
axes[1, 0].legend(fontsize=9)

axes[1, 1].plot(timeAxis[:800] * 1e6, ifSignal[:800], linewidth=1.4)
axes[1, 1].set_title('IF waveform after low-pass')
axes[1, 1].set_xlabel('Time [microsecond]')
axes[1, 1].set_ylabel('Amplitude')
axes[1, 1].grid(alpha=0.25)

plt.tight_layout()

print('混频后保留下来的 IF 主分量大约位于 8 MHz 和 18 MHz。')
print('这对应于 |92-100| MHz 和 |118-100| MHz 两个差频分量。')

到这里，模拟前端的主线已经完整了：天空信号先经历带通与时间增益调制，再叠加系统噪声，并经过放大、滤波与外差混频，最终以更适合 ADC 的频带与功率水平进入数字域。下一节我们就接着讨论这些中频信号如何被采样、如何通过 FFT/PFB 分道，以及相关器怎样把它们变成真正的可见度数据产品。

***

* 下一节： [7.4 数字相关器](7_4_digital.ipynb)